## 📑 Core Master Table
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AYDfEQ)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication  

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace
import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim
import boto3
import os

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

som_date = spark.sql(f'''select CAST(PreviousMonthLastWorkingDate AS DATE) from `{env_prefix}-cdh`.reference.ref_calendar where CalendarDate = {run_date!r}''').collect()[0]['PreviousMonthLastWorkingDate']  
som_date = som_date.strftime('%Y-%m-%d') 

# SOMCalendarreferenceDate = spark.sql(f'''SELECT date_add(last_day(add_months('{run_date}', -1)), 1) AS SOMCal''').collect()[0]['SOMCal']

soq_date = spark.sql(f'''
                    SELECT CASE WHEN month('{run_date}') IN (3,6,9,12) THEN last_day(add_months('{run_date}', -3))
                                WHEN month('{run_date}') IN (2,5,8,11) THEN last_day(add_months('{run_date}', -2)) 
                                WHEN month('{run_date}') IN (1,4,7,10) THEN last_day(add_months('{run_date}', -1))
                           END AS PrevQ_SOM
                         ''').collect()[0]['PrevQ_SOM']

EXPRT_MONTH = spark.sql(f'''SELECT date_format(last_day(add_months('{run_date}', -1)),'yyyyMM') AS Months''').collect()[0]['Months']

In [0]:
table_name = 'Core_MasterTable'

schema = (
    "`BusinessDate` date,"
    "`ExposureKey` string,"
    "`CDI` string,"
    "`PartyTaxRegistrationNumber` string,"
    "`TaxNumberStatic` string,"
    "`AgreementKey` string,"
    "`ContractUnionwise` string,"
    "`AssetSegment` string,"
    "`AccountStatus` string,"
    "`Bucket` string,"
    "`BucketCategoryNoMateriality` string,"
    "`BucketCategoryWithMateriality` string,"
    "`EBAStatus` string,"
    "`QueueDescription` string,"
    "`AgencyCode` string,"
    "`ProductCode` string,"
    "`SubproductCode` string,"
    "`OnBalanceAmount` string,"
    "`OffBalanceAmount` string,"
    "`FinalBalance` string,"
    "`Balance` string,"
    "`TotalBalance` string,"
    "`MLTAmount` string,"
    "`FrozenAmount` string,"
    "`DebtLPS` string,"
    "`PacketProcessType` string,"
    "`AccountCurrency` string,"
    "`ContractCurrency` int,"
    "`CustomerType` string,"
    "`ConnectedBorrowerFlag` string,"
    "`DeceasedFlag` string,"
    "`WatchlistFlag` string,"
    "`WatchlistDescription` string,"
    "`L3869MainBorrower` string,"
    "`L3869AllBeneficiaries` string,"
    "`Wholesale` string,"
    "`L3869Status` string,"
    "`L3869Decision` string,"
    "`L3869InterbankDescription` string,"
    "`CourtDate` string,"
    "`SubmissionDate` date,"
    "`OCWL4469Flag` string,"
    "`L4605Flag` string,"
    "`AutoLoan` string,"
    "`PrivateBanking` string,"
    "`StaffIndicator` string,"
    "`SpecialQueues` string,"
    "`CorporateBusinessCards` string,"
    "`GovernmentGuarantee` string,"
    "`BankruptcyFlag` string,"
    "`PrebankruptcyFlag` string,"
    "`MenuCode` string,"
    "`DenouncedDate` date,"
    "`NewDenouncementFlag` string,"
    "`LTV` string,"
    "`CollateralValue` decimal(19,2),"
    "`SecuredAccount` int,"
    "`SecuredDebtor` string,"
    "`SettlementType` string,"
    "`SettlementStatus` string,"
    "`CustomerN3869` string,"
    "`CustomerWatchlish` string,"
    "`Gemini` string,"
    "`SPV` string,"
    "`InfoInput` string,"
    "`DebtorOnBalanceTotalRetail` string,"
    "`DebtorBalanceTotalRetail` string,"
    "`DebtorTotalBalanceTotalRetail` string,"
    "`DebtorOnBalanceBU` string,"
    "`DebtorBalanceBU` string,"
    "`DebtorTotalBalanceBU` string,"
    "`ContractOnBalanceAmount` string,"
    "`ContractBalance` string,"
    "`ContractTotalBalance` string,"
    "`PaymentOrderCurrentStage` string,"
    "`PaymentOrderCurrentAgency` string,"
    "`RegularSuitCurrentStage` string,"
    "`RegularSuitCurrentAgency` string,"
    "`ForcedPrenotationCurrentStage` string,"
    "`ForcedPrenotationCurrentAgency` string,"
    "`ConfiscationCurrentStage` string,"
    "`ConfiscationCurrentAgency` string,"
    "`InsertTimeStamp` timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    som_date,  # type: ignore
)


### Tables from Databricks as Temps

In [0]:
QualcoPerimeterCalculated = spark.sql(
f'''
    SELECT de.*
         , p.BankingUnitCode SubProductInterestDescription
         , p.SubProduct SubProductDescription
        --  , ( SELECT max(fx.filled_fxrate)
        --      FROM  `p-bi`.common.fx_rates fx -- <-- need to change schema into dynamic parameter before deployment
        --      WHERE fx.OriginalCurrency = de.Currency
        --      AND fx.Date = de.BusinessDate
        --                     ) AS ExchangeRate
        , cast(null as double) as ExchangeRate
    FROM `{env_prefix}-cdh`.primary.exposure de
    LEFT JOIN `{env_prefix}-cdh`.primary.product p ON de.ProductKey = p.ProductKey 
                                        AND de.BusinessDate = p.BusinessDate
                                        AND de.SourceSystem = p.SourceSystem
    WHERE de.PortfolioName = 'Core Retail'
    AND de.BusinessDate = CAST('{som_date}' AS STRING)
    AND p.ProductDescription <> 'ΣΥΓΚ/ΣΗ ΑΠΟΣΒ ΟΦΕΙΛΩΝ ΚΑΤ/ΚΩΝ-ΧΩΡΙΣ HAIRCUT' 
    AND p.SubProduct <> 'ΒΟΗΘΗΤΙΚΟΙ ΛΟΓΑΡΙΑΣΜΟΙ'
    AND de.IsActiveFlag = TRUE
    AND de.PartyRelationshipType='Primary Owner'
'''
)
QualcoPerimeterCalculated.createOrReplaceTempView("QualcoPerimeterCalculated")

In [0]:
UnionTable = spark.sql(
f'''
    SELECT *
    FROM `{env_prefix}-cdh`.externalfiles.extunionfile
    WHERE ECAAssetClass IN ('Non-revolving Consumer','Residential','Revolving Consumer','SBPs') 
    AND BookingEntity IN ('AB1001','AB2063')
    AND SnapshotDate = last_day('{som_date}')
    AND to_date(ingestiontimestamp) = (SELECT MAX(to_date(ingestiontimestamp)) 
                        FROM `{env_prefix}-cdh`.externalfiles.extunionfile 
                        WHERE SnapshotDate = last_day('{som_date}'))
''')
UnionTable.createOrReplaceTempView("UnionTable")

previous_month_last_day = spark.sql(f'''SELECT last_day('{som_date}')''')
# Check if the dataset is empty and print a message accordingly
if UnionTable.count() == 0:
    print(f"The required external file extunionfile cataset for date {previous_month_last_day!r} is empty")

In [0]:
TotalAccountsPerimeter = spark.sql(
f'''
	SELECT *
	FROM
		(
			SELECT ExposureKey FROM QualcoPerimeterCalculated
			UNION
			SELECT ExposureKey FROM UnionTable
		) a
'''
)
TotalAccountsPerimeter.createOrReplaceTempView("TotalAccountsPerimeter")

In [0]:
MasterTableStatic = spark.sql(
f'''
    SELECT *
    FROM `{env_prefix}-reportinputfiles`.repoint.core_mastertablestatic 
'''
)
MasterTableStatic.createOrReplaceTempView("MasterTableStatic")

In [0]:
LatestUnion = spark.sql(
f'''
    SELECT *
    FROM `{env_prefix}-cdh`.externalfiles.extunionfile
    WHERE ECAAssetClass IN ('Non-revolving Consumer','Residential','Revolving Consumer','SBPs') 
    AND BookingEntity IN ('AB1001','AB2063')
    AND SnapshotDate = (CASE WHEN MONTH('{som_date}') IN (1,2,4,5,7,8,10,11) THEN '{soq_date}' ELSE last_day('{som_date}') END)
    AND to_date(ingestiontimestamp) = (SELECT MAX(to_date(ingestiontimestamp)) 
                                       FROM `{env_prefix}-cdh`.externalfiles.extunionfile 
                                       WHERE SnapshotDate = CASE WHEN MONTH('{som_date}') IN (1,2,4,5,7,8,10,11) THEN '{soq_date}' ELSE '{som_date}' END )
'''
)
LatestUnion.createOrReplaceTempView("LatestUnion")

In [0]:
QualcoEoM = spark.sql(
f'''
    SELECT B.ExposureKey
         , A.CIF
         , B.CDI
         , A.BRANCH_CODE
         , A.BRANCH_DESC
         , A.SETTLEMENT_TYPE
         , A.SETTLEMENT_CURRENT_STATUS
         , Customer_Watchlist
         , CustomerL3869
         , SPV
         , PRODUCT_CODE
    FROM `{env_prefix}-cdh-delivery`.repoint.cl_fullportfolioloansprm A
    INNER JOIN QualcoPerimeterCalculated B ON A.ACCOUNT_NUMBER=B.ExposureKey
    WHERE A.EXPRT_MONTH = '{EXPRT_MONTH}' AND 
          A.businessdate = (SELECT MAX(BusinessDate) 
                            FROM `{env_prefix}-cdh-delivery`.repoint.ml_fullportfolioprm
                            WHERE exprt_month='{EXPRT_MONTH}'
                           )
    
    -- (SELECT MAX(BusinessDate) 
    --                         FROM `{env_prefix}-cdh`.landed.repointqcab_cl_fullportfolioloansprm 
    --                         WHERE EXPRT_MONTH=CONCAT(YEAR('{run_date}'), lpad(month('{run_date}'), 2, '0'))
    --                        )
    UNION ALL

    SELECT B.ExposureKey
         , A.CIF
         , B.CDI
         , A.BRANCH_CODE
         , A.BRANCH_DESC
         , A.SETTLEMENT_TYPE
         , A.SETTLEMENT_CURRENT_STATUS
         , Customer_Watchlist
         , CustomerL3869
         , SPV
         , PRODUCT_CODE
    FROM `{env_prefix}-cdh-delivery`.repoint.cl_fullportfoliocardsprm A
    INNER JOIN QualcoPerimeterCalculated B ON A.ACCOUNT_NUMBER=B.ExposureKey
    WHERE A.EXPRT_MONTH = '{EXPRT_MONTH}' AND 
          A.businessdate = (SELECT MAX(BusinessDate) 
                            FROM `{env_prefix}-cdh-delivery`.repoint.ml_fullportfolioprm
                            WHERE exprt_month='{EXPRT_MONTH}'
                           )
    
--     (SELECT MAX(BusinessDate) 
--                             FROM `{env_prefix}-cdh`.landed.repointqcab_cl_fullportfoliocardsprm
--                             WHERE EXPRT_MONTH=CONCAT(YEAR('{run_date}'), lpad(month('{run_date}'), 2, '0'))
--                            )

    UNION ALL

    SELECT B.ExposureKey
         , A.CIF
         , B.CDI
         , A.BRANCH_CODE
         , A.BRANCH_DESC
         , A.SETTLEMENT_TYPE
         , A.SETTLEMENT_CURRENT_STATUS
         , Customer_Watchlist
         , CustomerL3869
         , SPV
         , PRODUCT_CODE
    FROM `{env_prefix}-cdh-delivery`.repoint.sb_fullportfolioprm A
    INNER JOIN QualcoPerimeterCalculated B ON A.ACCOUNT_NUMBER=B.ExposureKey
    WHERE A.EXPRT_MONTH = '{EXPRT_MONTH}' AND 
          A.businessdate = (SELECT MAX(BusinessDate) 
                            FROM `{env_prefix}-cdh-delivery`.repoint.ml_fullportfolioprm
                            WHERE exprt_month='{EXPRT_MONTH}'
                           )
    
--     (SELECT MAX(BusinessDate) 
--                             FROM `{env_prefix}-cdh`.landed.repointqcab_sb_fullportfolioprm
--                             WHERE EXPRT_MONTH=CONCAT(YEAR('{run_date}'), lpad(month('{run_date}'), 2, '0'))
--                            )

    UNION ALL

    SELECT B.ExposureKey
         , A.CIF
         , B.CDI
         , A.BRANCH_CODE
         , A.BRANCH_DESC
         , A.SETTLEMENT_TYPE
         , A.SETTLEMENT_CURRENT_STATUS
         , Customer_Watchlist
         , CustomerL3869
         , SPV
         , PRODUCT_CODE
    FROM `{env_prefix}-cdh-delivery`.repoint.ml_fullportfolioprm A
    INNER JOIN QualcoPerimeterCalculated B ON A.ACCOUNT_NUMBER=B.ExposureKey
    WHERE A.EXPRT_MONTH = '{EXPRT_MONTH}' AND 
          A.businessdate = (SELECT MAX(BusinessDate) 
                            FROM `{env_prefix}-cdh-delivery`.repoint.ml_fullportfolioprm
                            WHERE exprt_month='{EXPRT_MONTH}'
                           )
    
    -- (SELECT MAX(BusinessDate) 
    --                         FROM `{env_prefix}-cdh`.landed.repointqcab_ml_fullportfolioprm
    --                         WHERE EXPRT_MONTH=CONCAT(YEAR('{run_date}'), lpad(month('{run_date}'), 2, '0'))
    --                        )
'''
)
QualcoEoM.createOrReplaceTempView("QualcoEoM")

In [0]:
QualcoDaily = spark.sql(
f'''
    SELECT ACCOUNT_NUMBER
        , SPV_DESC
        , TOTAL_BALANCE_LCY
        , CASE_FROZEN_INTEREST_AMT_LCY
    FROM `{env_prefix}-cdh`.landed.repointqcab_daily_portfolio A
    INNER JOIN QualcoPerimeterCalculated B ON A.ACCOUNT_NUMBER = B.ExposureKey
    WHERE A.businessdate='{som_date}'
'''
)
QualcoDaily.createOrReplaceTempView("QualcoDaily")

# QualcoDaily.display(5)

In [0]:
LawMain_v0 = spark.sql(
f'''
    SELECT * FROM (
        SELECT  *
            ,    RANK() OVER (PARTITION BY CDI ORDER BY BusinessDate DESC) Rank
        FROM `{env_prefix}-cdh`.primary.underlaw
        WHERE PortfolioName = 'Core Retail'
        AND BusinessDate <= '{som_date}'
    ) a WHERE Rank = 1
'''
)
LawMain_v0.createOrReplaceTempView("LawMain_v0")

In [0]:
LawAll = spark.sql(
f'''
    SELECT DISTINCT e.cdi, e.ExposureKey
    FROM `{env_prefix}-cdh`.primary.exposure e
    INNER JOIN QualcoPerimeterCalculated B ON e.ExposureKey = B.ExposureKey AND e.BusinessDate = B.BusinessDate
    LEFT JOIN LawMain_v0 ul ON e.CDI = ul.CDI
    WHERE e.PortfolioName = 'Core Retail'
    AND e.IsActiveFlag = TRUE
    AND ul.InLawFlag = 1
'''
)
LawAll.createOrReplaceTempView("LawAll")

In [0]:
LawMain = spark.sql(
f'''
  SELECT DISTINCT 
              e.ExposureKey
            , ul.InLawFlag
            , ul.LawStatus
            , CASE WHEN ul.LawStatus	='Έφεση (Πρωτόδικη Απόφ. Δεκτή)' THEN 'Positive Court Decision' 
                                        ELSE ul.CalcLawStatus									
              END	AS LawDecision
            , ul.SubmissionDate
            , ul.CourtDate
            , CASE WHEN	ul.InterbankStatus3869 NOT IN 
                                    (
                                    'N/A'
                                    ,'Πελάτης Διατραπεζικής Ν3869 Follower'
                                    ,'Πελάτης Διατραπεζικής Ν3869 Leader'
                                    ,'Τειρεσίας - Παύση Εκκρεμοδικίας'
                                    ,'Έξοδος από Διατραπεζική ν.3869'
                                    )						   THEN 1 ELSE 0 END	AS L3869InterbankFlag
            ,CASE WHEN   ul.InterbankStatus3869 NOT IN 
                                    (
                                    'N/A'
                                    ,'Πελάτης Διατραπεζικής Ν3869 Follower'
                                    ,'Πελάτης Διατραπεζικής Ν3869 Leader'
                                    ,'Τειρεσίας - Παύση Εκκρεμοδικίας'
                                    ,'Έξοδος από Διατραπεζική ν.3869'
                                    )						   THEN ul.InterbankStatus3869 
                                                  ELSE NULL END	AS L3869InterbankDescription
            , ul.InterbankStatus3869 L3869InterbankDescription_2
  FROM QualcoPerimeterCalculated e
  LEFT JOIN LawMain_v0 ul ON e.CDI = ul.CDI
'''
)
LawMain.createOrReplaceTempView("LawMain")


In [0]:
ProptEliglbleStep1 = spark.sql(
f'''
    SELECT ExposureKey
            , MAX(LegalActionDate) LegalActivityDate
    FROM `{env_prefix}-cdh`.primary.legalaction
    WHERE LegalActionTypeCode LIKE '21%'
    AND   LegalActionTypeCode NOT IN ('2137','2139','2162','2146','2155','2164','2152','2156','2158','2165')
    AND   PortfolioName = 'Core Retail'
    AND   BusinessDate = '{som_date}'
    GROUP BY ExposureKey
'''
)
ProptEliglbleStep1.createOrReplaceTempView("ProptEliglbleStep1")

In [0]:
ProptNotEliglbleStep1 = spark.sql(
f'''
    SELECT ExposureKey
            , MAX(LegalActionDate) LegalActivityDate
    FROM `{env_prefix}-cdh`.primary.legalaction
    WHERE LegalActionTypeCode LIKE '21%'
    AND   LegalActionTypeCode IN ('2137','2139','2162','2146','2155','2164','2152','2156','2158','2165')
    AND   PortfolioName = 'Core Retail'
    AND   BusinessDate = '{som_date}'
    GROUP BY ExposureKey
'''
)
ProptNotEliglbleStep1.createOrReplaceTempView("ProptNotEliglbleStep1")

In [0]:
FinalEligiblePerimeter = spark.sql(
f'''
    SELECT DISTINCT 
        A.ExposureKey
    FROM ProptEliglbleStep1 A
    LEFT JOIN ProptNotEliglbleStep1	B ON A.ExposureKey=B.ExposureKey
    WHERE CASE WHEN A.LegalActivityDate> CASE WHEN B.LegalActivityDate IS NULL THEN '1900-01-01' ELSE B.LegalActivityDate END THEN 1 ELSE 0 END=1
'''
)
FinalEligiblePerimeter.createOrReplaceTempView("FinalEligiblePerimeter")

In [0]:
TotalPerimeter = spark.sql(
f'''
    SELECT e.ExposureKey
         , e.AgreementKey
         , e.CDI
         , e.PartyTaxRegistrationNumber
         , e.PartyRelationshipType
         , CASE WHEN c.ExposureKey IS NOT NULL THEN 1 ELSE 0 END ProptwxeytikoFlag
    FROM `{env_prefix}-cdh`.primary.exposure e
    LEFT JOIN FinalEligiblePerimeter c ON e.ExposureKey = c.ExposureKey   
    WHERE e.BusinessDate = '{som_date}'
    AND e.PortfolioName = 'Core Retail'
    AND e.IsActiveFlag = TRUE
    AND e.AssetSegment='SB'
'''
)
TotalPerimeter.createOrReplaceTempView("TotalPerimeter")

In [0]:
EligibleDebtor = spark.sql(
f'''
	SELECT DISTINCT 
           CDI
        ,  PartyTaxRegistrationNumber 
    FROM TotalPerimeter 
    WHERE ProptwxeytikoFlag=1

'''
)
EligibleDebtor.createOrReplaceTempView("EligibleDebtor")  

In [0]:
PreFinalStep1 = spark.sql(
f'''
    SELECT A.*
    FROM (
        SELECT A.* FROM TotalPerimeter A INNER JOIN EligibleDebtor B ON A.CDI=B.CDI
        
        UNION

        SELECT A.* FROM TotalPerimeter A INNER JOIN EligibleDebtor B ON A.PartyTaxRegistrationNumber=B.PartyTaxRegistrationNumber
        WHERE A.PartyTaxRegistrationNumber<>'000000000'
    ) A
   '''
)
PreFinalStep1.createOrReplaceTempView("PreFinalStep1")

In [0]:
PtwEliglbleStep1 = spark.sql(
f'''
    SELECT ExposureKey
            , MAX(LegalActionDate) LegalActivityDate
    FROM `{env_prefix}-cdh`.primary.legalaction
    WHERE LegalActionTypeCode LIKE '23%'
    AND   LegalActionTypeCode NOT IN ('2304','2308','2319')
    AND   PortfolioName = 'Core Retail'
    AND   BusinessDate = '{som_date}'
    GROUP BY ExposureKey
'''
)
PtwEliglbleStep1.createOrReplaceTempView("PtwEliglbleStep1")

In [0]:
PtwNotEliglbleStep1 = spark.sql(
f'''
    SELECT ExposureKey
            , MAX(LegalActionDate) LegalActivityDate
    FROM `{env_prefix}-cdh`.primary.legalaction
    WHERE LegalActionTypeCode LIKE '23%'
    AND   LegalActionTypeCode IN ('2304','2308','2319')
    AND   PortfolioName = 'Core Retail'
    AND   BusinessDate = '{som_date}'
    GROUP BY ExposureKey
'''
)
PtwNotEliglbleStep1.createOrReplaceTempView("PtwNotEliglbleStep1")

In [0]:
FinalEligiblePerimeterPtw = spark.sql(
f'''
    SELECT DISTINCT 
        A.ExposureKey
    FROM PtwEliglbleStep1 A
    LEFT JOIN PtwNotEliglbleStep1	B ON A.ExposureKey=B.ExposureKey
    WHERE CASE WHEN A.LegalActivityDate> CASE WHEN B.LegalActivityDate IS NULL THEN '1900-01-01' ELSE B.LegalActivityDate END THEN 1 ELSE 0 END=1
'''
)
FinalEligiblePerimeterPtw.createOrReplaceTempView("FinalEligiblePerimeterPtw")

In [0]:
TotalPerimeterPtw = spark.sql(
f'''
    SELECT e.ExposureKey
         , e.AgreementKey
         , e.CDI
         , e.PartyTaxRegistrationNumber
         , e.PartyRelationshipType
         , CASE WHEN c.ExposureKey IS NOT NULL THEN 1 ELSE 0 END ProptwxeytikoFlag
    FROM `{env_prefix}-cdh`.primary.exposure e
    LEFT JOIN FinalEligiblePerimeterPtw c ON e.ExposureKey = c.ExposureKey   
    WHERE e.BusinessDate = '{som_date}'
    AND e.PortfolioName = 'Core Retail'
    AND e.IsActiveFlag = TRUE
    AND e.AssetSegment='SB'
'''
)
TotalPerimeterPtw.createOrReplaceTempView("TotalPerimeterPtw")

In [0]:
EligibleDebtorPtw = spark.sql(
f'''
	SELECT DISTINCT 
           CDI
        ,  PartyTaxRegistrationNumber 
    FROM TotalPerimeterPtw 
    WHERE ProptwxeytikoFlag=1

'''
)
EligibleDebtorPtw.createOrReplaceTempView("EligibleDebtorPtw")

In [0]:
PreFinalStep1Ptw = spark.sql(
f'''
    SELECT DISTINCT A.*
    FROM (
        SELECT A.* FROM TotalPerimeterPtw A INNER JOIN EligibleDebtorPtw B ON A.CDI=B.CDI
        UNION

        SELECT A.* FROM TotalPerimeterPtw A INNER JOIN EligibleDebtorPtw B ON A.PartyTaxRegistrationNumber=B.PartyTaxRegistrationNumber
        WHERE A.PartyTaxRegistrationNumber<>'000000000' AND A.PartyTaxRegistrationNumber<>'' 
    ) A
   '''
)
PreFinalStep1Ptw.createOrReplaceTempView("PreFinalStep1Ptw")

In [0]:
FinalTable_Ptw = spark.sql(
f'''
  SELECT DISTINCT A.ExposureKey
		FROM PreFinalStep1Ptw A
  
'''
)
FinalTable_Ptw.createOrReplaceTempView("FinalTable_Ptw")

In [0]:
FinalTable_Pro = spark.sql(
f'''
    SELECT DISTINCT A.ExposureKey
    FROM PreFinalStep1 A
    LEFT JOIN FinalTable_Ptw D ON A.ExposureKey=D.ExposureKey
    WHERE A.PartyRelationshipType='Primary Owner'
    AND	  CASE WHEN D.ExposureKey IS NOT NULL THEN 1 ELSE 0 END=0
'''
)
FinalTable_Pro.createOrReplaceTempView("FinalTable_Pro")

In [0]:
Part1 = spark.sql(
f'''
SELECT 
DISTINCT CAST(A.ExposureKey AS STRING) AccountNumber
			,		B.Queue
			,		LPS_TBL.OperationalLegalStatusLevel3 AS LPS
			-- ,		A.LPS
			,		B.Agency
			,		L.PACKET_PROCESS_TYPE Packet
			,		A.AssetSegment
			,		CASE WHEN SubProductInterestDescription LIKE '25%'	THEN CAST(RIGHT(A.AgreementKey,LEN(a.AgreementKey)-2) AS VARCHAR(20))
						 WHEN SubProductInterestDescription LIKE '01%'	THEN CAST(RIGHT(A.AgreementKey,LEN(a.AgreementKey)-2) AS VARCHAR(20))
						 WHEN SubProductInterestDescription LIKE '23%'	THEN CAST(RIGHT(A.AgreementKey,LEN(a.AgreementKey)-2) AS VARCHAR(20))
						 WHEN SubProductInterestDescription LIKE '02%'	THEN CAST(RIGHT(A.AgreementKey,LEN(a.AgreementKey)-2) AS VARCHAR(20))
						 WHEN SubProductInterestDescription LIKE '36%'	THEN CAST(RIGHT(A.AgreementKey,LEN(a.AgreementKey)-2) AS VARCHAR(20))
						 WHEN SubProductInterestDescription LIKE '37%'	THEN CAST(RIGHT(A.AgreementKey,LEN(a.AgreementKey)-2) AS VARCHAR(20))
						 WHEN SubProductInterestDescription LIKE '18%'	THEN CAST(RIGHT(A.AgreementKey,LEN(a.AgreementKey)-2) AS VARCHAR(20))
						 WHEN SubProductInterestDescription LIKE '03%'	THEN CAST(RIGHT(A.AgreementKey,LEN(a.AgreementKey)-2) AS VARCHAR(20))
						 WHEN SubProductInterestDescription LIKE '%ΣΣΕ84%' AND LEN(a.AgreementKey)=12 THEN CAST(RIGHT(A.AgreementKey,10) AS VARCHAR(20))
						 WHEN SubProductInterestDescription LIKE '%ΣΣΕ84%' THEN CAST(A.AgreementKey AS VARCHAR(20))
						 WHEN LEN(A.AgreementKey)=10	THEN CAST(RIGHT(A.AgreementKey,LEN(A.AgreementKey)-2) AS VARCHAR(20))
						ELSE CAST(A.AgreementKey AS VARCHAR(20))
					END  AS AgreementNumber
			,		a.CDI
			,		COALESCE(CASE WHEN C.TaxRegistrationNumber IN ('','00000000','000000000','0000000000')					THEN NULL ELSE C.TaxRegistrationNumber END,
							 CASE WHEN D.MainBorrowerTaxIdentificationNumber IN ('','#N/A','00000000','000000000','0000000000')	THEN NULL ELSE D.MainBorrowerTaxIdentificationNumber END
							 ) AS TaxNumber
			,		COALESCE(CASE WHEN C.TaxRegistrationNumber IN ('','00000000','000000000','0000000000')					THEN NULL ELSE C.TaxRegistrationNumber END,		 CASE WHEN D.MainBorrowerTaxIdentificationNumber IN ('','#N/A','00000000','000000000','0000000000')	THEN NULL ELSE D.MainBorrowerTaxIdentificationNumber END,
			 				 CASE WHEN M.AFM='' THEN NULL ELSE M.AFM END
			 				 ) AS TaxNumberStatic
			,		CASE WHEN A.ExposureStatusCode='074-WRITE OFF'									THEN 99 
						 WHEN A.ExposureStatusCode='074-DENOUNCED' AND AssetSegment IN ('ML','SB') THEN 13
						 ELSE A.Bucket
					END AS Bucket
			,		CASE 
						WHEN SubProductDescription='ACBCL'					THEN 'A700'
						WHEN SubProductDescription='ACBDYN'					THEN 'DYN'
						WHEN SubProductDescription='ACBST'					THEN 'STORE'
						WHEN SubProductDescription='AMEX'					THEN 'AMEX'
						WHEN SubProductDescription='AMXPFS'					THEN 'AMXPFS'
						WHEN SubProductDescription='CRED'					THEN 'CRED'
						WHEN SubProductDescription='CREDC'					THEN 'CREDC'
						WHEN SubProductDescription='DINERS'					THEN 'DINERS'
						WHEN SubProductInterestDescription like '%ΣΣΕ84%'	THEN 'ΣΣΕ'
						WHEN SubProductInterestDescription like '%-%'		THEN CAST(CAST(SUBSTRING(SubProductInterestDescription, 0, CHARINDEX('-',SubProductInterestDescription)-1) AS INT) AS VARCHAR(10))
																			ELSE CAST(CAST(SubProductInterestDescription AS INT) AS VARCHAR(10))
					END	AS	ProductCodeFinal
			,		SubProductInterestDescription	AS SubproductCode
			,		CASE WHEN A.ExposureStatusCode='074-WRITE OFF' THEN ROUND(A.OffBalance/ (CASE WHEN A.ExchangeRate IS NULL THEN 1 ELSE A.ExchangeRate END),2) 
															ELSE ROUND(A.OutstandingTotalBalance/(CASE WHEN A.ExchangeRate IS NULL THEN 1 ELSE A.ExchangeRate END),2) 
					END AS balancecalc
			,		CASE WHEN D.DeNuncementFlag='1'		THEN 'Denounced'
						 WHEN A.ExposureStatusCode='074-WRITE OFF'		THEN 'Writeoff'
						 WHEN A.ExposureStatusCode='074-DENOUNCED'		THEN 'Denounced'
						 WHEN A.Bucket=0					THEN 'Current'
						 WHEN A.Bucket>0					THEN 'Non Denounced'
																ELSE 'N/A'
					END AS ArrearsStatus
			,		A.OutstandingTotalBalanceMLT TotalBalanceMLT
			,		A.OnBalance/coalesce(A.ExchangeRate,1)			OnBalanceAmount
			,		CASE WHEN SubProductInterestDescription like '%ΣΣΕ84%' THEN 0 ELSE A.OffBalance/coalesce(A.ExchangeRate,1)	END		OffBalanceAmount
			,		e.InLawFlag											LawMain
			,		CASE WHEN F.ExposureKey IS NOT NULL THEN 1 
														  ELSE 0 END	LawAll
			,		CASE WHEN B.Queue IN ('WHL_PRODUCTS','WHL_NOTICE ADMIN','WHL _SP TREATMENT','Wholesale_No_Flag',
										   'WHL _SP_TREATMENT_CL','Galaxy_SPV67_Wholesale_g','Cosmos_SPV73W_Wholesale_c') 
						 THEN 1 ELSE 0
					END whl
			,		TOTAL_BALANCE_LCY
			,		CASE_FROZEN_INTEREST_AMT_LCY
			,		CASE WHEN AssetSegment = 'CL' AND 
						(CASE WHEN SubProductInterestDescription LIKE '25%' THEN 'A700'
							 WHEN SubProductInterestDescription like '01%'		THEN 'AMEX'
							 WHEN SubProductInterestDescription like '23%'		THEN 'AMXPFS'
							 WHEN SubProductInterestDescription like '02%'		THEN 'CRED'
							 WHEN SubProductInterestDescription like '36%'		THEN 'CREDC'
							 WHEN SubProductInterestDescription like '37%'		THEN 'DINERS'
							 WHEN SubProductInterestDescription like '18%'		THEN 'DYN'
							 WHEN SubProductInterestDescription like '03%'		THEN 'STORE'
							 WHEN SubProductInterestDescription like '%ΣΣΕ84%'	THEN 'ΣΣΕ'
							 WHEN SubProductInterestDescription like '%-%'		THEN CAST(CAST(SUBSTRING(SubProductInterestDescription, 0, CHARINDEX('-',SubProductInterestDescription)) AS INT) AS VARCHAR(10))
							 ELSE CAST(CAST(SubProductInterestDescription AS INT) AS VARCHAR(10))
						END IN ('6016','6031','6900','6902','6903','6904','6905','6906','6907','6908','6909'))
					THEN 1 ELSE 0 
					END AS AutoLoan
			,		C.ConnectedBorrowerFlag --C2.ConnectedBorrowerFlag
			-- ,		CASE WHEN C.StaffFlag = '0' THEN 'False'
			-- 			 WHEN C.StaffFlag = '1' THEN 'True'
			-- 		END StaffFlag
			, 		C.StaffFlag
			,		CASE WHEN C.DeceasedFlag=TRUE THEN 1 ELSE 0 END DeceasedFlag
			,		B.WatchlistFlag
			,		A.Currency AS AccountCurrency
			,		CASE WHEN D.StateGuaranteeFlag IN ('1','2','3','4') OR A.IsGuaranteeFlag=TRUE  THEN 1 
																										ELSE 0 
					END GovernmentGuaranteeFlag
			,		CASE WHEN (H1.ExposureKey IS NOT NULL OR B.Queue='LEGAL - ΠΡΟΠΤΩΧΕΥΤΙΚΗ ΔΙΑΔΙΚΑΣΙΑ'  OR LPS_TBL.OperationalLegalStatusLevel3 IN ('ΣΥΜΦΩΝΙΑ ΣΥΝΔΙΑΛΛΑΓΗΣ','ΣΥΜΦΩΝΙΑ ΑΝΑΔΙΟΡΓΑΝΩΣΗΣ',
					 'ΜΗ ΣΥΜΦΩΝΙΑ ΕΚΚΑΘΑΡΙΣΗΣ','ΜΗ ΣΥΜΦΩΝΙΑ ΛΟΙΠΩΝ ΠΡΟΠΤΩΧΕΥΤΙΚΩΝ ΔΙΑΔΙΚΑΣΙΩΝ','ΜΗ ΣΥΜΦΩΝΙΑ ΣΥΝΔΙΑΛΛΑΓΗΣ','ΜΗ ΣΥΜΦΩΝΙΑ ΑΝΑΔΙΟΡΓΑΝΩΣΗΣ','ΑΙΤΗΜΑ ΕΙΔΙΚΗΣ ΕΚΚΑΘΑΡΙΣΗΣ',
					 'ΑΙΤΗΜΑ ΛΟΙΠΩΝ ΠΡΟΠΤΩΧΕΥΤΙΚΩΝ ΔΙΑΔΙΚΑΣΙΩΝ','ΑΙΤΗΜΑ ΣΥΝΔΙΑΛΛΑΓΗΣ',				 'ΑΙΤΗΜΑ ΑΝΑΔΙΟΡΓΑΝΩΣΗΣ','ΣΥΜΦΩΝΙΑ ΕΚΚΑΘΑΡΙΣΗΣ','ΣΥΜΦΩΝΙΑ ΛΟΙΠΩΝ ΠΡΟΠΤΩΧΕΥΤΙΚΩΝ ΔΙΑΔΙΚΑΣΙΩΝ')
							  ) AND AssetSegment='SB'
					THEN 1 
					ELSE 0 
					END AS Proptwxeytiko
			,		CASE WHEN (H2.ExposureKey IS NOT NULL OR B.Queue='LEGAL - ΠΤΩΧΕΥΣΗ ΕΠΙΧΕΙΡΗΣΕΩΝ' 	OR LPS_TBL.OperationalLegalStatusLevel3 IN ('ΕΚΠΟΙΗΣΗ ΠΕΡΙΟΥΣΙΑΚΩΝ','ΠΕΡΑΤΩΣΗ ΑΚΑΡΠΗ',
					 'ΜΗ ΕΠΑΛΗΘΕΥΣΗ ΑΠΑΙΤΗΣΕΩΝ ΤΡΑΠΕΖΗΣ','ΕΝΩΣΗ ΠΙΣΤΩΤΩΝ','ΑΠΟΡΡΙΨΗ ΑΙΤΗΣΗΣ ΠΤΩΧΕΥΣΗΣ','ΑΙΤΗΣΗ ΠΤΩΧΕΥΣΗΣ')
							  )
					THEN 1 
					ELSE 0 
					END AS Ptwxeysh
			,		c.PartyType CustomerType
			,		CASE WHEN SubProductInterestDescription LIKE '25%'	AND Assetsegment='CL'	THEN '25'
						 WHEN SubProductInterestDescription LIKE '01%'	AND Assetsegment='CL'	THEN '01'
						 WHEN SubProductInterestDescription LIKE '23%'	AND Assetsegment='CL'	THEN '23'
						 WHEN SubProductInterestDescription LIKE '02%'	AND Assetsegment='CL'	THEN '02'
						 WHEN SubProductInterestDescription LIKE '36%'	AND Assetsegment='CL'	THEN '36'
						 WHEN SubProductInterestDescription LIKE '37%'	AND Assetsegment='CL'	THEN '37'
						 WHEN SubProductInterestDescription LIKE '18%'	AND Assetsegment='CL'	THEN '18'
						 WHEN SubProductInterestDescription LIKE '03%'	AND Assetsegment='CL'	THEN '03'
																								ELSE NULL 
					END AS MenuCode
			,		D.EBAStatus 
			,		D.BucketCategoryNMateriality
			,		D.BucketCategoryWithMateriality
			,		A.DenouncedDate
			,	 	COALESCE(D.ltv,K.LTV) LTV 
			,		E.LawStatus
			,		e.LawDecision
			,		E.L3869InterbankDescription_2
			,		E.CourtDate
			,		E.SubmissionDate			,		CASE WHEN B.Queue LIKE '%4469%' THEN 1 ELSE 0 END  AS L4469
			,		c.N4605Flag
   			,		CASE WHEN MONTH(a.BusinessDate) IN (1,2,4,5,7,8,10,11) THEN CAST(replace(K.CollateralCapTotalTangible, ',', '') AS DECIMAL(10,2))
					ELSE CAST(replace(D.CollateralCapTotalTangible, ',', '') AS DECIMAL(10,2))		
    				END AS CollateralValue
  			,		CASE WHEN 
			 			CASE WHEN MONTH(a.BusinessDate) IN (1,2,4,5,7,8,10,11) THEN CAST(replace(K.CollateralCapTotalTangible, ',', '') AS DECIMAL(10,2))
			 																ELSE CAST(replace(D.CollateralCapTotalTangible, ',', '') AS DECIMAL(10,2))
			 			END>0 
			 		THEN 1 ELSE 0 
			 		END AS SecuredAccount
			 ,		CASE WHEN I.BRANCH_CODE='0950'	THEN 1 
													ELSE 0 
					END AS PrivateBanking
			 ,		I.SETTLEMENT_TYPE
			 ,		I.SETTLEMENT_CURRENT_STATUS	
			 ,		I.CustomerL3869
			 ,		I.Customer_Watchlist
			 ,		CASE WHEN D.SecuritizationCode='13'	  THEN 1
						 WHEN G.SPV_DESC='GEMINI CORE SECURITISATION DAC' THEN 1
																		  ELSE 0 
					END Gemini
			 ,		G.SPV_DESC AS spv
			 ,		CASE WHEN D.ExposureKey IS NOT NULL THEN 'Both' 
													  ELSE 'Qc' 
					END InfoInput
			 ,		B.CurrentStagePaymentOrder			AS PaymentOrderCurrentStage
			 ,		B.PaymentOrderCurrentAgency
			 ,		B.CurrentStageRegularSuit			AS RegularSuitCurrentStage
			 ,		B.RegularSuitCurrentAgency
			 ,		B.CurrentStageForcedPrenotation		AS ForcedPrenotationCurrentStage
			 ,		B.ForcedPrenotationCurrentAgency
			 ,		B.CurrentStageConfiscation			AS ConfiscationCurrentStage
			 ,		B.ConfiscationCurrentAgency
 			 ,		NULL AS CorporateBusinessCards
FROM QualcoPerimeterCalculated A
		LEFT JOIN `{env_prefix}-cdh`.derivative.dlegalprocessstatusagreementdebtor LPS_TBL ON LPS_TBL.AgreementKey = A.AgreementKey AND LPS_TBL.CDI = A.CDI AND LPS_TBL.BusinessDate = A.BusinessDate
		LEFT JOIN `{env_prefix}-cdh`.primary.casecode 	B ON A.CaseCode = B.CaseCodeKey AND B.BusinessDate = A.BusinessDate AND B.PortfolioName='Core Retail'
		LEFT JOIN `{env_prefix}-cdh`.primary.party		C ON A.CDI = C.CDI AND C.BusinessDate = A.BusinessDate AND C.SourceSystem='qca' AND C.PortfolioName = 'Core Retail'
		LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.core_retailprimary L ON A.ExposureKey=L.ACCOUNT_NUMBER AND L.BusinessDate = '{run_date}' AND L.EXPRT_MONTH = '{EXPRT_MONTH}' 
		LEFT JOIN UnionTable                                D  ON A.ExposureKey = D.ExposureNumber
		LEFT JOIN LawMain                                   E  ON A.ExposureKey = E.ExposureKey
		LEFT JOIN LawAll                                    F  ON A.ExposureKey = F.ExposureKey
		LEFT JOIN QualcoDaily                               G  ON A.ExposureKey = G.ACCOUNT_NUMBER
		LEFT JOIN FinalTable_Pro                            H1 ON A.ExposureKey = H1.ExposureKey
		LEFT JOIN FinalTable_Ptw                            H2 ON A.ExposureKey = H2.ExposureKey
		LEFT JOIN QualcoEoM                                 I  ON A.ExposureKey = I.ExposureKey
		LEFT JOIN LatestUnion                               K  ON A.ExposureKey = K.ExposureNumber
		LEFT JOIN MasterTableStatic              M  ON A.ExposureKey=M.Account_Number
'''
)
Part1.createOrReplaceTempView("Part1")

In [0]:
Part2 = spark.sql(
f'''
SELECT 
					CAST(A.ExposureNumber AS STRING) AccountNumber
			,		NULL as Queue
			,		NULL as LPS 
			,		NULL AS AgencyCode
			,		NULL AS Packet
			,		CASE WHEN A.ECAAssetClass='Non-revolving Consumer'	THEN 'CL'
						 WHEN A.ECAAssetClass='Revolving Consumer'		THEN 'CL'
						 WHEN A.ECAAssetClass='Residential'			THEN 'ML'
						 WHEN A.ECAAssetClass='SBPs'					THEN 'SB'
					END AS AssetSegment
			,		CASE 
						WHEN A.AgreementKey='#N/A'																					THEN NULL
						WHEN A.ECAAssetClass IN ('Revolving Consumer')																THEN RIGHT(A.AgreementKey,LEN(A.AgreementKey)-2)
						WHEN A.ECAAssetClass IN ('Non-revolving Consumer') AND LEN(A.AgreementKey)=15								THEN A.AgreementKey
						WHEN A.ECAAssetClass IN ('Non-revolving Consumer')															THEN CONCAT('00',A.AgreementKey)
						WHEN A.ExposureSourceSystemKey='SSE'  AND LEN(A.AgreementKey)=10										THEN CAST(A.AgreementKey AS VARCHAR(10))
						WHEN A.ECAAssetClass in ('Residential','SBPs') AND LEN(A.AgreementKey)=10 AND LEFT(A.AgreementKey,1)<>'0'THEN CONCAT('00',A.AgreementKey)
						WHEN A.ECAAssetClass in ('Residential','SBPs') AND LEN(A.AgreementKey)=10									THEN A.AgreementKey
						WHEN A.ECAAssetClass in ('Residential','SBPs') AND LEN(A.AgreementKey)=9									THEN CONCAT('0',A.AgreementKey)
																																	ELSE A.AgreementKey
						END AgreementNumber
			,		CASE WHEN LEN(A.BorrowerKey)=3 THEN CONCAT('0000000',A.BorrowerKey)
						 WHEN LEN(A.BorrowerKey)=4 THEN CONCAT('000000' ,A.BorrowerKey)
						 WHEN LEN(A.BorrowerKey)=5 THEN CONCAT('00000'  ,A.BorrowerKey)
						 WHEN LEN(A.BorrowerKey)=6 THEN CONCAT('0000'   ,A.BorrowerKey)
						 WHEN LEN(A.BorrowerKey)=7 THEN CONCAT('000'    ,A.BorrowerKey)
						 WHEN LEN(A.BorrowerKey)=8 THEN CONCAT('00'     ,A.BorrowerKey)
						 WHEN LEN(A.BorrowerKey)=9 THEN CONCAT('0'      ,A.BorrowerKey)
												  ELSE					 A.BorrowerKey
					END CDI
			,		CASE WHEN A.MainBorrowerTaxIdentificationNumber IN ('','#N/A','00000000','000000000','0000000000') THEN NULL 
																									 ELSE A.MainBorrowerTaxIdentificationNumber 
					END TaxNumber
			,		COALESCE(
			 		CASE WHEN A.MainBorrowerTaxIdentificationNumber IN ('','#N/A','00000000','000000000','0000000000') THEN NULL 
						 ELSE A.MainBorrowerTaxIdentificationNumber 
			 		END,
			 		CASE WHEN M.AFM='' THEN NULL ELSE M.AFM END
					)	TaxNumberStatic
			,		CAST(A.BucketCategoryNMateriality AS INT) AS Bucket
			,		CASE WHEN A.ExposureSourceSystemKey='SSE'										THEN 'ΣΣΕ'
						 WHEN A.ECAAssetClass NOT IN ('Revolving Consumer')							THEN A.ProductCode 
						 WHEN A.AgreementKey like '25%'	AND A.ECAAssetClass ='Revolving Consumer'	THEN 'A700'
						 WHEN A.AgreementKey like '01%'	AND A.ECAAssetClass ='Revolving Consumer'	THEN 'AMEX'
						 WHEN A.AgreementKey like '23%'	AND A.ECAAssetClass ='Revolving Consumer'	THEN 'AMXPFS'
						 WHEN A.AgreementKey like '02%'	AND A.ECAAssetClass ='Revolving Consumer'	THEN 'CRED'
						 WHEN A.AgreementKey like '36%'	AND A.ECAAssetClass ='Revolving Consumer'	THEN 'CREDC'
						 WHEN A.AgreementKey like '37%'	AND A.ECAAssetClass ='Revolving Consumer'	THEN 'DINERS'
						 WHEN A.AgreementKey like '18%'	AND A.ECAAssetClass ='Revolving Consumer'	THEN 'DYN'
						 WHEN A.AgreementKey like '03%'	AND A.ECAAssetClass ='Revolving Consumer'	THEN 'STORE'
																									ELSE NULL
					END ProductCodeFinal
			,		CASE WHEN a.ExposureSourceSystemKey IN ('TRD')	OR A.ECAAssetClass='Residential' THEN NULL 
																									 ELSE A.ProductCode 
					END SubproductCode
			,		CAST(REPLACE(a.OnbalanceAmount,',','') AS DECIMAL(19,2)) balancecalc
			,		CASE WHEN A.DeNuncementFlag ='1'		THEN 'Denounced' 
						 WHEN A.BucketCategoryNMateriality ='0' THEN 'Current'
																ELSE 'non denounced'
					END ArrearsStatus
			,		CAST(REPLACE(a.OnbalanceAmount,',','') AS DECIMAL(19,2))+CAST(REPLACE(a.OffBalanceAmount,',','') AS DECIMAL(19,2)) TotalBalanceMLT
			,		CAST(REPLACE(a.OnbalanceAmount,',','') AS DECIMAL(19,2)) OnBalanceAmount
			,		0 OffBalanceAmount
			,		NULL LawMain
			,		NULL LawAll
			,		NULL whl
			,		a.OnbalanceAmount TOTAL_BALANCE_LCY
			,		NULL CASE_FROZEN_INTEREST_AMT_LCY
			,		CASE WHEN  A.ECAAssetClass IN ('Non-revolving Consumer','Revolving Consumer') AND
							CASE WHEN A.ExposureSourceSystemKey='SSE' THEN 'ΣΣΕ'
								 WHEN A.ExposureKey LIKE '25%'		  THEN 'A700'
								 WHEN A.ExposureKey LIKE '01%'		  THEN 'AMEX'
								 WHEN A.ExposureKey LIKE '23%'		  THEN 'AMXPFS'
								 WHEN A.ExposureKey LIKE '02%'		  THEN 'CRED'
								 WHEN A.ExposureKey LIKE '36%'		  THEN 'CREDC'
								 WHEN A.ExposureKey LIKE '37%'		  THEN 'DINERS'
								 WHEN A.ExposureKey LIKE '18%'		  THEN 'DYN'
								 WHEN A.ExposureKey LIKE '03%'		  THEN 'STORE'
																	  ELSE A.ProductCode 
							END IN ('6016','6031','6900','6902','6903','6904','6905','6906','6907','6908','6909') 
					THEN 1 ELSE 0 
					END	AS AutoLoan
			,		C.ConnectedBorrowerFlag 
			,		NULL StaffFlag
			,		NULL DeceasedFlag
			,		NULL WatchlistFlag
			,		A.ExposureCurrency AccountCurrency
			,		CASE WHEN A.StateGuaranteeFlag IN ('1','2','3','4') THEN 1 ELSE 0 END GovernmentGuaranteeFlag
			,		NULL Proptwxeytiko
			,		NULL Ptwxeysh
			,		NULL CustomerType
			,		CASE 
						WHEN A.AgreementKey like '25%'	AND A.ECAAssetClass ='Revolving Consumer'	THEN '25'
						WHEN A.AgreementKey like '01%'	AND A.ECAAssetClass ='Revolving Consumer'	THEN '01'
						WHEN A.AgreementKey like '23%'	AND A.ECAAssetClass ='Revolving Consumer'	THEN '23'
						WHEN A.AgreementKey like '02%'	AND A.ECAAssetClass ='Revolving Consumer'	THEN '02'
						WHEN A.AgreementKey like '36%'	AND A.ECAAssetClass ='Revolving Consumer'	THEN '36'
						WHEN A.AgreementKey like '37%'	AND A.ECAAssetClass ='Revolving Consumer'	THEN '37'
						WHEN A.AgreementKey like '18%'	AND A.ECAAssetClass ='Revolving Consumer'	THEN '18'
						WHEN A.AgreementKey like '03%'	AND A.ECAAssetClass ='Revolving Consumer'	THEN '03'
						ELSE NULL
					END MenuCode
			,		A.EBAStatus 
			,		A.BucketCategoryNMateriality
			,		A.BucketCategoryWithMateriality
			,		NULL DenouncedDate
			,		COALESCE(A.LTV,D.LTV) LTV
			/***,	 	COALESCE(A.LTV,D.LTV,E.LTV)) LTV ***/ --will be included after calc_data.RA_Modifications deployment	
			,		NULL LawStatus
			,		NULL LawDecision
			,		NULL L3869InterbankDescription_2
			,		NULL CourtDate
			,		NULL SubmissionDate
			,		NULL L4469
			,		NULL N4605Flag
    		,		CASE WHEN MONTH(CAST(A.SnapshotDate AS DATE)) IN (1,2,4,5,7,8,10,11) THEN CAST(replace(D.CollateralCapTotalTangible, ',', '') AS DECIMAL(19,2))
																			ELSE CAST(replace(A.CollateralCapTotalTangible, ',', '') AS DECIMAL(19,2))
					END AS CollateralValue
			,		CASE WHEN 
						CASE WHEN MONTH(CAST(A.SnapshotDate AS DATE)) IN (1,2,4,5,7,8,10,11)  THEN CAST(replace(D.CollateralCapTotalTangible, ',', '') AS DECIMAL(19,2))
																				ELSE CAST(replace(A.CollateralCapTotalTangible, ',', '') AS DECIMAL(19,2))
						END>0 
					THEN 1 
					ELSE 0 
				END SecuredAccount
			 ,		CASE WHEN A.AcMaintenServiceBranch=950 THEN 1 ELSE 0 END AS PrivateBanking
			 ,		NULL SETTLEMENT_TYPE
			 ,		NULL SETTLEMENT_CURRENT_STATUS
			 ,		NULL `Customer N3869`
			 ,		NULL CUSTOMER_WATCHLIST
			 ,		CASE WHEN A.SecuritizationCode='13' THEN 1 ELSE 0 END Gemini
			 ,		A.NonCoreFlag AS spv
			 ,		'Union' as InfoInput
			 ,		NULL PaymentOrderCurrentStage
			 ,		NULL PaymentOrderCurrentAgency
			 ,		NULL RegularSuitCurrentStage
			 ,		NULL RegularSuitCurrentAgency
			 ,		NULL ForcedPrenotationCurrentStage
			 ,		NULL ForcedPrenotationCurrentAgency
			 ,		NULL ConfiscationCurrentStage
			 ,		NULL ConfiscationCurrentAgency
 			 ,		NULL AS CorporateBusinessCards
		FROM		UnionTable												A 
		LEFT JOIN   QualcoPerimeterCalculated							B ON A.ExposureNumber=B.ExposureKey
		LEFT JOIN   `{env_prefix}-cdh`.primary.party					C ON
					CASE WHEN LEN(A.BorrowerKey)=3 THEN CONCAT('0000000',A.BorrowerKey)
						 WHEN LEN(A.BorrowerKey)=4 THEN CONCAT('000000' ,A.BorrowerKey)
						 WHEN LEN(A.BorrowerKey)=5 THEN CONCAT('00000'  ,A.BorrowerKey)
						 WHEN LEN(A.BorrowerKey)=6 THEN CONCAT('0000'   ,A.BorrowerKey)
						 WHEN LEN(A.BorrowerKey)=7 THEN CONCAT('000'    ,A.BorrowerKey)
						 WHEN LEN(A.BorrowerKey)=8 THEN CONCAT('00'     ,A.BorrowerKey)
						 WHEN LEN(A.BorrowerKey)=9 THEN CONCAT('0'      ,A.BorrowerKey)
						 ELSE     A.BorrowerKey
						 END	=	C.CDI 
					AND CAST(C.BusinessDate AS STRING)=CAST('{som_date}' AS STRING) AND C.SourceSystem='qca'
		LEFT JOIN LatestUnion											D ON A.ExposureNumber=D.ExposureNumber
		LEFT JOIN MasterTableStatic										M ON A.ExposureNumber=M.Account_Number
		WHERE B.ExposureKey IS NULL
'''
)
Part2.createOrReplaceTempView("Part2")

In [0]:
Part3 = spark.sql(
f'''
SELECT 
				A.AccountNumber
			,	A.CDI
			,	CASE WHEN A.TaxNumber			='N/A'	THEN NULL ELSE A.TaxNumber			END TaxNumber
			,	CASE WHEN A.TaxNumberStatic		='N/A'	THEN NULL ELSE A.TaxNumberStatic	END TaxNumberStatic
      		, 	A.AgreementNumber
			,	A.ArrearsStatus
			,	CAST(a.balancecalc AS DECIMAL(19,2))										AS	Balance
			,	CASE WHEN ProductCodeFinal IN ('4609','4612','4613')		   
													     THEN 'ML' ELSE AssetSegment	    END	AS AssetSegment
			,	CASE WHEN A.Queue				=''		 THEN NULL ELSE A.Queue			END Queues
			,	A.Agency
			,	CASE WHEN A.SubproductCode		=''		 THEN NULL ELSE a.SubproductCode	END SubproductCode
			,	CASE WHEN A.LPS					=''		 THEN NULL ELSE A.LPS				END LPS

			,	CASE WHEN A.Packet	=''		 THEN NULL ELSE A.Packet	END PacketProcessType
			,	a.Bucket
			,	null														 				AS LegalAgencyAnathesis
			,	CASE WHEN A.ProductCodeFinal	=''		THEN NULL ELSE A.ProductCodeFinal	END ProductCode
			,	CAST(TotalBalanceMLT AS DECIMAL(19,2)) AS TotalBalanceMLT
			,	OnBalanceAmount
			,	OffBalanceAmount
			,	a.LawMain
			,	a.LawAll
			,	a.whl
   			,	CAST(replace(TOTAL_BALANCE_LCY, ',', '') AS DECIMAL(19,2)) AS TOTAL_BALANCE_LCY
			-- ,	TOTAL_BALANCE_LCY
			,	CASE_FROZEN_INTEREST_AMT_LCY
			,	a.AutoLoan
			,	a.ConnectedBorrowerFlag
			,	a.StaffFlag ABStaffFlag
			,	a.DeceasedFlag
			,	case when WatchlistFlag is not null and WatchlistFlag<>'' and WatchlistFlag not like 'N4605%'  then 1 else 0 end WatchlistFlag
			,   case when WatchlistFlag is not null and WatchlistFlag<>'' and WatchlistFlag not like 'N4605%'  then 1 else 0 end WatchlistFlagContract
			,	case when WatchlistFlag is not null and WatchlistFlag<>'' and WatchlistFlag not like 'N4605%'  then WatchlistFlag else NULL end WatchlistFlagDescription
			,	case when A.Queue in ('Emp_Confirmed_Fraud','Emp_Υλοποιημένες Συμβιβαστικές Εξοφλήσεις') then A.Queue else null end as SpecialQueues
			,	a.AccountCurrency
			,	a.AccountCurrency ContractCurrency
			,	A.CorporateBusinessCards AS CorporateBusinessCard
			,	GovernmentGuaranteeFlag
			,	Proptwxeytiko
			,	Ptwxeysh
			,	case when a.CustomerType is null and a.Taxnumber like '09%' then 'Organization'
					 when a.CustomerType is null and a.Taxnumber like '08%' then 'Organization'
					 when a.CustomerType is null and a.Taxnumber like '8%'  then 'Organization'
					 when a.CustomerType is null and a.Taxnumber like '9%'  then 'Organization'
					 when a.CustomerType is not null then a.CustomerType
					 else 'Individual' end CustomerType
			,	A.MenuCode
			,	A.EBAStatus EBA_Status
			,	A.BucketCategoryNMateriality
			,	A.BucketCategoryWithMateriality
			,	case when a.ArrearsStatus in ('Denounced','Writeoff') then a.DenouncedDate else null end DenouncedDate
			,	case when 
				case when a.ArrearsStatus in ('Denounced') then last_day(a.DenouncedDate) else null end='{som_date}'
				then 'True' else 'False' end
				NewDenounced
			,	A.LTV
			,	A.LawStatus
			,	A.LawDecision
			,	A.L3869InterbankDescription_2 L3869InterbankDescription
			,	A.CourtDate
			,	A.SubmissionDate
			,	A.L4469
			,	A.N4605Flag
			,	A.CollateralValue
			,	A.SecuredAccount
			,	A.SecuredAccount AS SecuredCustomer
			,	A.PrivateBanking
			,	A.SETTLEMENT_TYPE			AS SettlementType
			,	A.SETTLEMENT_CURRENT_STATUS AS SettlementStatus
			,	A.CustomerL3869
			,	A.Customer_Watchlist
			,	A.Gemini
			,	A.spv
			,	A.InfoInput
			,	A.PaymentOrderCurrentStage
			,	A.PaymentOrderCurrentAgency
			,	A.RegularSuitCurrentStage
			,	A.RegularSuitCurrentAgency
			,	A.ForcedPrenotationCurrentStage
			,	A.ForcedPrenotationCurrentAgency
			,	A.ConfiscationCurrentStage
			,	A.ConfiscationCurrentAgency
FROM (
	SELECT * 
	FROM Part1
	UNION ALL
	SELECT *
	FROM Part2
) A
LEFT JOIN MasterTableStatic B ON A.AccountNumber = B.Account_Number 
'''
)
Part3.createOrReplaceTempView("Part3")

In [0]:
Part4 = spark.sql(
f'''
select A.*,
				CASE WHEN A.AgreementNumber IS NOT NULL THEN CONCAT(CASE WHEN MenuCode IS NOT NULL THEN MenuCode ELSE "" END, A.AgreementNumber)
    				 WHEN A.AgreementNumber IS NULL AND B.Contract_Unionwise IS NULL THEN NULL
					 ELSE B.Contract_Unionwise 					  
      			END ContractUnionwise
from Part3 A
LEFT JOIN MasterTableStatic B ON A.AccountNumber = B.Account_Number 

'''
)
Part4.createOrReplaceTempView("Part4")


In [0]:
TaxNumberAggregations = spark.sql(
f'''
SELECT  TaxNumber
      , MAX(CASE WHEN whl=1 THEN 1 ELSE 0 END)                                                                       AS whlAFM
      , MAX(CASE WHEN ConnectedBorrowerFlag=TRUE THEN 1 ELSE 0 END)                                                  AS ConnectedBorrowerAFM
      , MAX(CASE WHEN ABStaffFlag='1' AND InfoInput IN ('Both','Qc') THEN 1 ELSE 0 END)                                           AS ABStaffAFM
      , MAX(CASE WHEN PrivateBanking=1 THEN 1 ELSE 0 END)                                                                       AS PrivateBankingAFM	
      , MAX(CASE WHEN DeceasedFlag=1 OR Queues IN ('Emp_Confirmed Deceased','All_Borrowers_Deceased') THEN 1 ELSE 0 END)        AS DeceasedAFM		
      , MAX(CASE WHEN Proptwxeytiko=1 AND AssetSegment = 'SB' THEN 1 ELSE 0 END)                                                AS PropAFM
      , MAX(CASE WHEN Ptwxeysh=1 THEN 1 ELSE 0 END)                                                                  AS ptwAFM
      , MAX(CASE WHEN L4469=1 THEN 1 ELSE 0 END)                                                                                AS L4469AFM
      , MAX(CASE WHEN N4605Flag=1 THEN 1 ELSE 0 END)                                                                            AS N4605AFM
      , MAX(CASE WHEN SecuredAccount=1 THEN 1 ELSE 0 END)                                                                       AS SecuredAFM
FROM Part4
GROUP BY TaxNumber
'''
)
TaxNumberAggregations.createOrReplaceTempView("TaxNumberAggregations")

In [0]:
CDIAggregations = spark.sql(
f'''
SELECT  CDI
      , MAX(CASE WHEN whl=1 THEN 1 ELSE 0 END)                                                                     AS whlCDI
      , MAX(CASE WHEN ConnectedBorrowerFlag=TRUE THEN 1 ELSE 0 END)                                                AS ConnectedBorrowerCDI
      , MAX(CASE WHEN ABStaffFlag='1' AND InfoInput IN ('Both','Qc') THEN 1 ELSE 0 END)                                         AS ABStaffCDI
      , MAX(CASE WHEN PrivateBanking=1 THEN 1 ELSE 0 END)                                                                     AS PrivateBankingCDI	
      , MAX(CASE WHEN DeceasedFlag=1 OR Queues IN ('Emp_Confirmed Deceased','All_Borrowers_Deceased') THEN 1 ELSE 0 END)      AS DeceasedCDI		
      , MAX(CASE WHEN Proptwxeytiko=1 AND AssetSegment = 'SB' THEN 1 ELSE 0 END)                                              AS PropCDI
      , MAX(CASE WHEN Ptwxeysh=1 THEN 1 ELSE 0 END)                                                                AS ptwCDI
      , MAX(CASE WHEN L4469=1 THEN 1 ELSE 0 END)                                                                              AS L4469CDI
      , MAX(CASE WHEN N4605Flag=1 THEN 1 ELSE 0 END)                                                                          AS N4605CDI
      , MAX(CASE WHEN SecuredAccount=1 THEN 1 ELSE 0 END)                                                                     AS SecuredCDI
FROM Part4
GROUP BY CDI
'''
)
CDIAggregations.createOrReplaceTempView("CDIAggregations")

In [0]:
AGRAggregations = spark.sql(
f'''
SELECT  ContractUnionwise
      , MAX(CASE WHEN (AccountCurrency<>'EUR' AND AccountCurrency<>'ΕUR') AND AgreementNumber IS NOT NULL THEN 1 END) AS CurrencyAGR
      , MAX(CASE WHEN WatchlistFlag=1 AND InfoInput IN ('Both','Qc') THEN 1 END) AS WatchlistAGR
      , MAX(CASE WHEN GovernmentGuaranteeFlag=1 AND AgreementNumber<>'' AND AgreementNumber IS NOT NULL THEN 1 ELSE 0 END) AS GovGuarAGR
FROM Part4
GROUP BY ContractUnionwise
'''
)
AGRAggregations.createOrReplaceTempView("AGRAggregations")

In [0]:
CustomerTotal = spark.sql(
f'''
SELECT		TaxNumber
			,		SUM(TotalBalanceMLT) TotalBalance
			,		sum(Balance)		 Balance 
			,		sum(OnBalanceAmount) OnBalanceAmount
FROM Part4
WHERE TaxNumber IS NOT NULL
GROUP BY TaxNumber
'''
)
CustomerTotal.createOrReplaceTempView("CustomerTotal")

In [0]:
CustomerPerBU = spark.sql(
f'''
		SELECT		TaxNumber,AssetSegment
			,		SUM(TotalBalanceMLT) TotalBalance
			,		sum(Balance)		 Balance 
			,		sum(OnBalanceAmount) OnBalanceAmount
		FROM Part4
		WHERE TaxNumber IS NOT NULL
		GROUP BY TaxNumber,AssetSegment
'''
)
CustomerPerBU.createOrReplaceTempView("CustomerPerBU")

In [0]:
CustomerTotalCDI = spark.sql(
f'''
		SELECT		CDI
			,		SUM(TotalBalanceMLT) TotalBalance
			,		sum(Balance)		 Balance 
			,		sum(OnBalanceAmount) OnBalanceAmount
		FROM Part4
		--WHERE TaxNumber IS NOT NULL
		GROUP BY CDI
'''
)
CustomerTotalCDI.createOrReplaceTempView("CustomerTotalCDI")

In [0]:
CustomerPerBUCDI = spark.sql(
f'''
		SELECT		CDI,AssetSegment
			,		SUM(TotalBalanceMLT) TotalBalance
			,		sum(Balance)		 Balance 
			,		sum(OnBalanceAmount) OnBalanceAmount
		FROM Part4
		GROUP BY CDI,AssetSegment
'''
)
CustomerPerBUCDI.createOrReplaceTempView("CustomerPerBUCDI")

In [0]:
ContractTotal = spark.sql(
f'''
		SELECT		ContractUnionwise
			,		SUM(TotalBalanceMLT) TotalBalance
			,		sum(Balance)		 Balance 
			,		sum(OnBalanceAmount) OnBalanceAmount
		FROM Part4
		GROUP BY ContractUnionwise
'''
)
ContractTotal.createOrReplaceTempView("ContractTotal")

In [0]:
FinalTable = spark.sql(
f'''
SELECT DISTINCT
        	'{som_date}' ReferenceDate
		    ,	CAST(A.AccountNumber AS STRING) AccountNumber
		    ,	CAST(A.CDI AS STRING) DebtorCDI
		    ,	CAST(A.TaxNumber AS STRING) TaxNumber
		    ,	CAST(A.TaxNumberStatic AS STRING) TaxNumberStatic
		    ,	CAST(A.AgreementNumber AS STRING) AgreementNumber
		    ,	CAST(A.ContractUnionwise AS STRING) ContractUnionwise
		    ,	CAST(A.AssetSegment AS STRING) AssetSegment
		    ,	CAST(A.ArrearsStatus AS STRING) AS AccountStatus
		    ,	CAST(A.Bucket AS STRING) Bucket
		    ,	CAST(A.BucketCategoryNMateriality AS STRING) BucketCategoryNoMateriality
			,	CAST(A.BucketCategoryWithMateriality AS STRING) BucketCategoryWithMateriality
			,	CAST(A.EBA_Status AS STRING) EBAStatus
			,	CAST(A.Queues  AS STRING) QueueDescription
			,	CAST(A.Agency AS STRING) AgencyCode 
			, 	CASE WHEN A.InfoInput='Union' AND J.ECAAssetClass='Revolving Consumer' THEN
					 CASE WHEN A.ContractUnionwise IS NULL			THEN J.ProductCode 
						  WHEN A.ContractUnionwise LIKE '25%'		THEN 'A700'
						  WHEN A.ContractUnionwise LIKE '01%'		THEN 'AMEX'
						  WHEN A.ContractUnionwise LIKE '23%'		THEN 'AMXPFS'
						  WHEN A.ContractUnionwise LIKE '02%'		THEN 'CRED'
						  WHEN A.ContractUnionwise LIKE '36%'		THEN 'CREDC'
						  WHEN A.ContractUnionwise LIKE '37%'		THEN 'DINERS'
						  WHEN A.ContractUnionwise LIKE '18%'		THEN 'DYN'
						  WHEN A.ContractUnionwise LIKE '03%'		THEN 'STORE'
						END	ELSE A.ProductCode 
      			END ProductCode
			,	CAST(A.SubproductCode AS STRING) SubproductCode
			,	CAST(A.OnBalanceAmount AS STRING) OnBalanceAmount
			,	CAST(A.OffBalanceAmount AS STRING) OffBalanceAmount 
			,	CAST(A.Balance AS STRING)								AS FinalBalance
			,	CAST(A.TOTAL_BALANCE_LCY AS STRING)						AS Balance
			,	CAST(A.TotalBalanceMLT AS STRING)						AS TotalBalance
			,	CAST(A.TotalBalanceMLT-A.TOTAL_BALANCE_LCY AS STRING)	AS MLTAmount
			,	CAST(A.CASE_FROZEN_INTEREST_AMT_LCY AS STRING)			AS FrozenAmount
			,	CAST(A.LPS AS STRING) DebtLPS
			,	CAST(A.PacketProcessType AS STRING) PacketProcessType
			,	CAST(A.AccountCurrency AS STRING) AccountCurrency
			, 	CASE WHEN I.CurrencyAGR = 1 THEN 1
					 WHEN ContractCurrency = 'EUR' OR ContractCurrency = 'ΕUR' THEN 0 
   					 ELSE 1  
        		END AS ContractCurrency
			,	CAST(A.CustomerType AS STRING) CustomerType
			, 	CAST(  CASE WHEN G.ConnectedBorrowerAFM = 1 OR H.ConnectedBorrowerCDI = 1 THEN 1
                    		WHEN ConnectedBorrowerFlag = TRUE THEN 1
                      		WHEN ConnectedBorrowerFlag = FALSE THEN 0
                        	ELSE 0 END AS STRING)	                    AS ConnectedBorrowerFlag 
			, 	CAST(  CASE WHEN G.DeceasedAFM = 1 OR H.DeceasedCDI = 1 THEN 1 
   					   ELSE CASE WHEN A.DeceasedFlag IS NULL THEN 0 ELSE A.DeceasedFlag END END AS STRING)      				AS DeceasedFlag
			-- ,	CAST(CASE WHEN I.WatchlistAGR  = 1 AND InfoInput IN ('Both','QC') THEN 1 ELSE A.WatchlistFlagContract END AS STRING) AS WatchlistFlag --IL: KEEP COMMENT
			--,	CAST(CASE WHEN I.WatchlistAGR  = 1 AND InfoInput IN ('Both','QC') AND (A.WatchlistFlagDescription = '' OR A.WatchlistFlagDescription IS NULL) THEN 'YesContract' ELSE A.WatchlistFlagDescription END AS STRING) AS WatchlistDescription   --IL: KEEP COMMENT
   			,	CAST(CASE WHEN I.WatchlistAGR  = 1 THEN 1 ELSE A.WatchlistFlagContract END AS STRING) AS WatchlistFlag
			,	CAST(A.WatchlistFlagDescription AS STRING) AS WatchlistDescription   
			,	CAST(A.LawMain AS STRING)		AS L3869MainBorrower
			,	CAST(A.LawAll	 AS STRING) AS L3869AllBeneficiaries
			,	CAST(CASE WHEN G.whlAFM = 1 OR H.whlCDI = 1 THEN 1  
                    ELSE CASE WHEN A.whl IS NULL THEN 0 ELSE A.whl END END AS STRING)		AS Wholesale
			,	CAST(A.LawStatus AS STRING) L3869Status
			,	CAST(A.LawDecision AS STRING) L3869Decision
			,	CAST(A.L3869InterbankDescription AS STRING) L3869InterbankDescription
			,	CAST(A.CourtDate AS STRING) CourtDate
			,	CAST(A.SubmissionDate AS STRING) SubmissionDate
			,	CAST(CASE WHEN G.L4469AFM = 1 OR H.L4469CDI = 1 THEN 1
						  ELSE CASE WHEN A.L4469 IS NULL THEN 0 ELSE A.L4469 END END  AS STRING)              						AS OCWL4469Flag
			,	CAST(CASE WHEN G.N4605AFM = 1 OR H.N4605CDI = 1 THEN 1
                    	  ELSE CASE WHEN A.N4605Flag IS NULL THEN 0 ELSE A.N4605Flag END END  AS STRING)          					AS L4605Flag
			,	CAST(A.AutoLoan AS STRING) AutoLoan
			,	CAST(CASE WHEN G.PrivateBankingAFM = 1 OR H.PrivateBankingCDI = 1 THEN 1 
                    ELSE A.PrivateBanking END AS STRING) AS PrivateBanking
			,	CAST(CASE WHEN (G.ABStaffAFM =1 OR H.ABStaffCDI =1) AND InfoInput IN ('Both','Qc') THEN 1
                      WHEN A.ABStaffFlag IS NULL OR A.ABStaffFlag = '' THEN 0 ELSE CAST(A.ABStaffFlag AS INT) END   AS STRING)     	AS StaffIndicator 
            --         	  WHEN A.ABStaffFlag = '1' THEN 1 ELSE 0 END   AS STRING)     					AS StaffIndicator --IL: KEEP
			,	CAST(A.SpecialQueues AS STRING) SpecialQueues
			,	'' AS CorporateBusinessCards
			--,	CAST(A.GovernmentGuaranteeFlag AS STRING) GovernmentGuarantee --IL:KEEP
			,	CAST(CASE WHEN I.GovGuarAGR  = 1 THEN 1 ELSE A.GovernmentGuaranteeFlag END AS STRING) AS GovernmentGuarantee
			,	CAST(CASE WHEN G.ptwAFM = 1 OR H.ptwCDI = 1 THEN 1 
                    	  ELSE CASE WHEN A.Ptwxeysh IS NULL THEN 0 ELSE A.Ptwxeysh END END AS STRING)           				AS BankruptcyFlag
			,	CAST(CASE WHEN A.AssetSegment='SB' AND (G.PropAFM = 1 OR H.PropCDI = 1) THEN 1
                    	  ELSE CASE WHEN A.Proptwxeytiko IS NULL THEN 0 ELSE A.Proptwxeytiko END END AS STRING)     				AS PrebankruptcyFlag
			, 	CASE WHEN A.InfoInput='Union' AND J.ECAAssetClass='Revolving Consumer' THEN
					 CASE WHEN A.ContractUnionwise LIKE '25%'		THEN '25'
						  WHEN A.ContractUnionwise LIKE '01%'		THEN '01'
						  WHEN A.ContractUnionwise LIKE '23%'		THEN '23'
						  WHEN A.ContractUnionwise LIKE '02%'		THEN '02'
						  WHEN A.ContractUnionwise LIKE '36%'		THEN '36'
						  WHEN A.ContractUnionwise LIKE '37%'		THEN '37'
						  WHEN A.ContractUnionwise LIKE '18%'		THEN '18'
						  WHEN A.ContractUnionwise LIKE '03%'		THEN '03'
						  ELSE NULL
						END	ELSE A.MenuCode END MenuCode			
			,	CAST(A.DenouncedDate AS STRING) DenouncedDate
			,	CAST(A.NewDenounced AS STRING) NewDenouncementFlag
			,	CAST(A.LTV AS STRING) LTV
   			,	CAST(A.CollateralValue AS DECIMAL(19,2)) CollateralValue
   			,	CAST(A.SecuredAccount AS INT) SecuredAccount
			,	CAST(CASE WHEN G.SecuredAFM=1 OR H.SecuredCDI=1 THEN 1 
                  ELSE A.SecuredCustomer END AS STRING) AS SecuredDebtor
			,	CAST(A.SettlementType AS STRING) SettlementType
			,	CAST(A.SettlementStatus AS STRING) SettlementStatus
			,	CAST(A.CustomerL3869 AS STRING)	AS CustomerN3869
			,	CAST(A.Customer_Watchlist  AS STRING) AS CustomerWatchlish
			,	CAST(A.Gemini AS STRING) Gemini
			,	CAST(CASE WHEN A.spv IN ('GEMINI CORE SECURITISATION DAC','Orion X Securitisation DAC','COSMOS SECURITISATION DAC','Galaxy II Funding DAC') THEN NULL ELSE A.SPV END AS STRING) AS SPV
			,	CAST(A.InfoInput AS STRING) InfoInput
			,	CAST(COALESCE(B.OnBalanceAmount	    ,C.OnBalanceAmount	    ) AS STRING)	AS DebtorOnBalanceTotalRetail
			,	CAST(COALESCE(B.Balance			    ,C.Balance			    ) AS STRING)	AS DebtorBalanceTotalRetail
			,	CAST(COALESCE(B.TotalBalance		,C.TotalBalance		    ) AS STRING) 	AS DebtorTotalBalanceTotalRetail
			,	CAST(COALESCE(D.OnBalanceAmount	    ,E.OnBalanceAmount	    ) AS STRING)	AS DebtorOnBalanceBU
			,	CAST(COALESCE(D.Balance			    ,E.Balance			    ) AS STRING)	AS DebtorBalanceBU
			,	CAST(COALESCE(D.TotalBalance		,E.TotalBalance		    ) AS STRING)	AS DebtorTotalBalanceBU
			,	CAST(F.OnBalanceAmount 										AS STRING)		AS ContractOnBalanceAmount
			,	CAST(F.Balance 												AS STRING)		AS ContractBalance
			,	CAST(F.TotalBalance 										AS STRING)		AS ContractTotalBalance
			,	CAST(A.PaymentOrderCurrentStage 							AS STRING) PaymentOrderCurrentStage
			,	CAST(A.PaymentOrderCurrentAgency 							AS STRING) PaymentOrderCurrentAgency
			,	CAST(A.RegularSuitCurrentStage 								AS STRING) RegularSuitCurrentStage
			,	CAST(A.RegularSuitCurrentAgency 							AS STRING) RegularSuitCurrentAgency
			,	CAST(A.ForcedPrenotationCurrentStage 						AS STRING) ForcedPrenotationCurrentStage
			,	CAST(A.ForcedPrenotationCurrentAgency 						AS STRING) ForcedPrenotationCurrentAgency
			,	CAST(A.ConfiscationCurrentStage 							AS STRING) ConfiscationCurrentStage
			,	CAST(A.ConfiscationCurrentAgency 							AS STRING) ConfiscationCurrentAgency
FROM Part4 A
LEFT JOIN CustomerTotal		    B ON A.TaxNumber=B.TaxNumber
LEFT JOIN CustomerTotalCDI		C ON A.CDI=C.CDI
LEFT JOIN CustomerPerBU		    D ON A.TaxNumber=D.TaxNumber AND A.AssetSegment=D.AssetSegment
LEFT JOIN CustomerPerBUCDI		E ON A.CDI=E.CDI AND A.AssetSegment=E.AssetSegment
LEFT JOIN ContractTotal		    F ON A.ContractUnionwise=F.ContractUnionwise
LEFT JOIN TaxNumberAggregations G ON A.TaxNumber=G.TaxNumber
LEFT JOIN CDIAggregations       H ON A.CDI=H.CDI
LEFT JOIN AGRAggregations       I ON A.ContractUnionwise=I.ContractUnionwise
LEFT JOIN UnionTable 			J ON A.AccountNumber=J.ExposureNumber
'''
)
FinalTable.createOrReplaceTempView("FinalTable")

In [0]:
final_df = spark.sql(
f'''
SELECT 
  CAST(ReferenceDate AS DATE) AS BusinessDate,
  CAST(AccountNumber AS string) AS ExposureKey,
  CAST(DebtorCDI AS string) AS CDI,
  CAST(TaxNumber AS string) AS PartyTaxRegistrationNumber,
  CAST(TaxNumberStatic AS string) AS TaxNumberStatic,
  CAST(AgreementNumber AS string) AS AgreementKey,
  CAST(ContractUnionwise AS string) AS ContractUnionwise,
  CAST(AssetSegment AS string) AS AssetSegment,
  CAST(AccountStatus AS string) AS AccountStatus,
  CAST(Bucket AS string) AS Bucket,
  CAST(BucketCategoryNoMateriality AS string) AS BucketCategoryNoMateriality,
  CAST(BucketCategoryWithMateriality AS string) AS BucketCategoryWithMateriality,
  CAST(EBAStatus AS string) AS EBAStatus,
  CAST(QueueDescription AS string) AS QueueDescription,
  CAST(AgencyCode AS string) AS AgencyCode,
  CAST(ProductCode AS string) AS ProductCode,
  CAST(SubproductCode AS string) AS SubproductCode,
  CAST(OnBalanceAmount AS string) AS OnBalanceAmount,
  CAST(OffBalanceAmount AS string) AS OffBalanceAmount,
  CAST(FinalBalance AS string) AS FinalBalance,
  CAST(Balance AS string) AS Balance,
  CAST(TotalBalance AS string) AS TotalBalance,
  CAST(MLTAmount AS string) AS MLTAmount,
  CAST(FrozenAmount AS string) AS FrozenAmount,
  CAST(DebtLPS AS string) AS DebtLPS,
  CAST(PacketProcessType AS string) AS PacketProcessType,
  CAST(AccountCurrency AS string) AS AccountCurrency,
  CAST(ContractCurrency AS int) AS ContractCurrency,
  CAST(CustomerType AS string) AS CustomerType,
  CAST(ConnectedBorrowerFlag AS string) AS ConnectedBorrowerFlag,
  CAST(DeceasedFlag AS string) AS DeceasedFlag,
  CAST(WatchlistFlag AS string) AS WatchlistFlag,
  CAST(WatchlistDescription AS string) AS WatchlistDescription,
  CAST(L3869MainBorrower AS string) AS L3869MainBorrower,
  CAST(L3869AllBeneficiaries AS string) AS L3869AllBeneficiaries,
  CAST(Wholesale AS string) AS Wholesale,
  CAST(L3869Status AS string) AS L3869Status,
  CAST(L3869Decision AS string) AS L3869Decision,
  CAST(L3869InterbankDescription AS string) AS L3869InterbankDescription,
  CAST(CourtDate AS string) AS CourtDate,
  CAST(SubmissionDate AS DATE) AS SubmissionDate,--
  CAST(OCWL4469Flag AS string) AS OCWL4469Flag,
  CAST(L4605Flag AS string) AS L4605Flag,
  CAST(AutoLoan AS string) AS AutoLoan,
  CAST(PrivateBanking AS string) AS PrivateBanking,
  CAST(StaffIndicator AS string) AS StaffIndicator,
  CAST(SpecialQueues AS string) AS SpecialQueues,
  CAST(CorporateBusinessCards AS string) AS CorporateBusinessCards,
  CAST(GovernmentGuarantee AS string) AS GovernmentGuarantee,
  CAST(BankruptcyFlag AS string) AS BankruptcyFlag,
  CAST(PrebankruptcyFlag AS string) AS PrebankruptcyFlag,
  CAST(MenuCode AS string) AS MenuCode,
  CAST(DenouncedDate AS DATE) AS DenouncedDate,--
  CAST(NewDenouncementFlag AS string) AS NewDenouncementFlag,
  CAST(LTV AS string) AS LTV,
  CAST(CollateralValue AS decimal(19,2)) AS CollateralValue,
  CAST(SecuredAccount AS int) AS SecuredAccount,
  CAST(SecuredDebtor AS string) AS SecuredDebtor,
  CAST(SettlementType AS string) AS SettlementType,
  CAST(SettlementStatus AS string) AS SettlementStatus,
  CAST(CustomerN3869 AS string) AS CustomerN3869,
  CAST(CustomerWatchlish AS string) AS CustomerWatchlish,
  CAST(Gemini AS string) AS Gemini,
  CAST(SPV AS string) AS SPV,
  CAST(InfoInput AS string) AS InfoInput,
  CAST(DebtorOnBalanceTotalRetail AS string) AS DebtorOnBalanceTotalRetail,
  CAST(DebtorBalanceTotalRetail AS string) AS DebtorBalanceTotalRetail,
  CAST(DebtorTotalBalanceTotalRetail AS string) AS DebtorTotalBalanceTotalRetail,
  CAST(DebtorOnBalanceBU AS string) AS DebtorOnBalanceBU,
  CAST(DebtorBalanceBU AS string) AS DebtorBalanceBU,
  CAST(DebtorTotalBalanceBU AS string) AS DebtorTotalBalanceBU,
  CAST(ContractOnBalanceAmount AS string) AS ContractOnBalanceAmount,
  CAST(ContractBalance AS string) AS ContractBalance,
  CAST(ContractTotalBalance AS string) AS ContractTotalBalance,
  CAST(PaymentOrderCurrentStage AS string) AS PaymentOrderCurrentStage,
  CAST(PaymentOrderCurrentAgency AS string) AS PaymentOrderCurrentAgency,
  CAST(RegularSuitCurrentStage AS string) AS RegularSuitCurrentStage,
  CAST(RegularSuitCurrentAgency AS string) AS RegularSuitCurrentAgency,
  CAST(ForcedPrenotationCurrentStage AS string) AS ForcedPrenotationCurrentStage,
  CAST(ForcedPrenotationCurrentAgency AS string) AS ForcedPrenotationCurrentAgency,
  CAST(ConfiscationCurrentStage AS string) AS ConfiscationCurrentStage,
  CAST(ConfiscationCurrentAgency AS string) AS ConfiscationCurrentAgency,
  CAST(getdate() AS TIMESTAMP) AS InsertTimeStamp
FROM FinalTable A
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_MasterTable: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey"]
check_table(delivery_table_path, som_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
MasterTable = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = '{som_date}'
'''
)
MasterTable.createOrReplaceTempView("MasterTable")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/MasterTable/"
file_name = "MasterTable_" +som_date+ ".csv"
MasterTable.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)